# FOVAL HPO on Google Colab

**Setup:**
1. Runtime: **T4 GPU**
2. Upload `foval_hpo.zip` to your Google Drive root folder
3. Run all cells

In [ ]:
# Force re-extract
!rm -rf /content/foval
!unzip -o /content/drive/MyDrive/FOVAL_HPO/foval_hpo.zip -d /content/foval
%cd /content/foval
!grep "load_study\|create_study" hpo.py


In [ ]:
# 1. Mount Google Drive
from google.colab import drive, auth
auth.authenticate_user()
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 2. Extract project from Drive
!unzip -o /content/drive/MyDrive/FOVAL_HPO/foval_hpo.zip -d /content/foval
%cd /content/foval

Archive:  /content/drive/MyDrive/FOVAL_HPO/foval_hpo.zip
  inflating: /content/foval/hpo.py   
  inflating: /content/foval/main.py  
 extracting: /content/foval/data/__init__.py  
  inflating: /content/foval/data/AbstractDatasetClass.py  
  inflating: /content/foval/data/PreprocessedDataset.py  
  inflating: /content/foval/data/foval_preprocessor.py  
  inflating: /content/foval/data/utilities.py  
  inflating: /content/foval/data/giw_dataset.py  
  inflating: /content/foval/data/robustVision_dataset.py  
  inflating: /content/foval/data/TuftsDataset.py  
  inflating: /content/foval/data/MixedDatasetClass.py  
  inflating: /content/foval/data/SpecificMixDatasetClass.py  
  inflating: /content/foval/data/preprocessed/giw.parquet  
  inflating: /content/foval/data/preprocessed/robustvision.parquet  
  inflating: /content/foval/data/preprocessed/tufts.parquet  
  inflating: /content/foval/models/foval.py  
  inflating: /content/foval/models/config/foval.json  
  inflating: /content/foval/

In [ ]:
# 3. Install dependencies
!pip install -q optuna pyarrow

In [ ]:
# 4. Verify GPU + data
import torch, os
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()
for f in sorted(os.listdir('data/preprocessed')):
    print(f"  {f}: {os.path.getsize(f'data/preprocessed/{f}') / 1024:.0f} KB")

CUDA: True
GPU: NVIDIA A100-SXM4-40GB

  giw.parquet: 20473 KB
  robustvision.parquet: 1017 KB
  tufts.parquet: 88 KB


In [ ]:
# 5. Run HPO
N_TRIALS = 20
!python -W ignore hpo.py --n_trials {N_TRIALS} --resume

Traceback (most recent call last):
  File "/content/foval/hpo.py", line 230, in <module>
    main()
  File "/content/foval/hpo.py", line 197, in main
    study = optuna.load_study(study_name=STUDY_NAME, storage=storage)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/optuna/_convert_positional_args.py", line 127, in converter_wrapper
    return func(**kwargs)  # type: ignore[call-arg]
           ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/study.py", line 1403, in load_study
    study = Study(study_name=study_name, storage=storage, sampler=sampler, pruner=pruner)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/study.py", line 88, in __init__
    study_id = storage.get_study_id_from_name(study_name)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3

In [ ]:
# 6. Save results back to Drive
!cp hpo_results.db /content/drive/MyDrive/FOVAL_HPO/
!cp models/config/foval_hpo_best.json /content/drive/MyDrive/FOVAL_HPO/
print('Results saved to Google Drive/FOVAL_HPO/')

cp: cannot stat 'models/config/foval_hpo_best.json': No such file or directory
Results saved to Google Drive/FOVAL_HPO/


In [ ]:
# 7. Show results
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.load_study(study_name='foval_hpo', storage='sqlite:///hpo_results.db')
print(f"Total trials: {len(study.trials)}")
print(f"Best MAE: {study.best_value:.2f} cm")
print(f"\nBest params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

print(f"\nTop 5 trials:")
trials = sorted(study.trials, key=lambda t: t.value if t.value else float('inf'))
for t in trials[:5]:
    print(f"  Trial {t.number}: MAE={t.value:.2f} cm")

KeyError: 'Record does not exist.'